# Align

## Imports

In [1]:
import os

import numpy as np
import pandas as pd
from scipy.spatial import KDTree

import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
import plotly.graph_objs as go
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

## Functions

In [2]:
def rigid_align(points_ref, points_matched):
    centroid_ref = np.mean(points_ref, axis=0)
    centroid_target = np.mean(points_matched, axis=0)
    points_ref_centered = points_ref - centroid_ref
    points_target_centered = points_matched - centroid_target
    H = np.dot(points_target_centered.T, points_ref_centered)
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = np.dot(Vt.T, U.T)
    t = centroid_ref - np.dot(R, centroid_target)
    return R, t

def icp_align(points_ref, points_target, max_iter=100, tol=1e-6):
    tree = KDTree(points_ref)
    prev_error = float('inf')
    R = np.eye(3)
    t = np.zeros(3)
    current_target = points_target.copy()
    
    for _ in range(max_iter):
        dist, idx = tree.query(current_target)
        points_matched_ref = points_ref[idx]
        
        # Filter exception matching
        mask = dist < np.median(dist) * 3
        if np.sum(mask) < 3:
            break
        points_matched_ref = points_matched_ref[mask]
        current_target_masked = current_target[mask]
        
        iter_R, iter_t = rigid_align(points_matched_ref, current_target_masked)
        
        R = np.dot(iter_R, R)
        t = np.dot(iter_R, t) + iter_t
        current_target = np.dot(iter_R, current_target.T).T + iter_t
        
        error = np.mean(dist[mask])
        if abs(prev_error - error) < tol:
            break
        prev_error = error
    
    return R, t

## Pipeline

In [3]:
# Parameters setting
CROP_FRAMES = 26993
MODE = "2D" # 2D or 3D

In [4]:
# Project data folder
os.chdir("/home/Sol/Documents/Wirefree_Miniscope_3D_Staircase/Data/Behavior_Data/101F copy/batch3")

In [5]:
# Set file paths
if MODE == "2D":
    file_paths = ['2D and 3D_recording_day1_04062021/2D/trial1/final_behavior.csv', '2D and 3D_recording_day2_04072021/2D/trial1/final_behavior.csv', '2D and 3D_recording_day3_04082021/2D/trial1/final_behavior.csv', '2D and 3D_recording_day4_04092021/2D/trial1/final_behavior.csv', '2D and 3D_recording_day5_04102021/2D/trial1/final_behavior.csv']
else:
    file_paths = ['2D and 3D_recording_day1_04062021/3D/trial1/final_behavior.csv', '2D and 3D_recording_day2_04072021/3D/trial1/final_behavior.csv', '2D and 3D_recording_day3_04082021/3D/trial1/final_behavior.csv', '2D and 3D_recording_day4_04092021/3D/trial1/final_behavior.csv', '2D and 3D_recording_day5_04102021/3D/trial1/final_behavior.csv']

In [6]:
df_ref = pd.read_csv(file_paths[0]).iloc[:CROP_FRAMES]
points_ref = df_ref[['x', 'y', 'z']].values  # Extract the first 

# Start combining
df_combined = df_ref.copy()

for file_path in file_paths[1:]:
    df_target = pd.read_csv(file_path)
    df_target = df_target.iloc[:CROP_FRAMES]
    
    # Abstract point cloud and filter
    points_target = df_target[['x', 'y', 'z']].dropna().values

    if len(points_target) == 0:
        print(f"Skipping {file_path}: no valid points")
        continue
    
    # Use ICP to align
    R, t = icp_align(points_ref, points_target)
    
    # Align x, y, z
    points_aligned = np.dot(R, points_target.T).T + t
    df_target[['x', 'y', 'z']] = points_aligned
    
    # Spin x and y
    if 'dx' in df_target.columns and 'dy' in df_target.columns:
        vectors = df_target[['dx', 'dy']].values
        vectors_3d = np.column_stack([vectors, np.zeros(len(vectors))])
        vectors_aligned = np.dot(R, vectors_3d.T).T[:, :2]
        df_target[['dx', 'dy']] = vectors_aligned
        
    # Add NaN Between 2 df to remove the link lines
    if not df_combined.empty:
        insert_row = df_combined.iloc[-1].copy()
        insert_row['x'] = np.nan
        insert_row['y'] = np.nan
        insert_row['z'] = np.nan
        df_combined = pd.concat([df_combined, pd.DataFrame([insert_row])], ignore_index=True)
    
    df_combined = pd.concat([df_combined, df_target], ignore_index=True)

df_combined.to_csv('combined_aligned_trajectories.csv', index=False)
print("Combined and aligned trajectories saved to 'combined_aligned_trajectories.csv'")

Combined and aligned trajectories saved to 'combined_aligned_trajectories.csv'


In [7]:
def plot_3d_trajectory_with_direction(df, name):
# Normalize direction to 0-1 for colormap
    dir_norm = df['direction'] / 360.0

    # Get colors from matplotlib's rainbow colormap
    cmap = cm.get_cmap('rainbow')
    colors_rgba = cmap(dir_norm)

    # Normalize speed
    speed_norm = df['speed'] / df['speed'].max() if df['speed'].max() > 0 else df['speed']

    # Apply gamma correction using speed_norm as gamma
    colors_gamma = []
    for i, color in enumerate(colors_rgba):
        gamma = 1 - speed_norm.iloc[i] if speed_norm.iloc[i] > 0 else 1.0  # Avoid gamma=0
        r, g, b, a = color
        r = r ** gamma
        g = g ** gamma
        b = b ** gamma
        colors_gamma.append(rgb2hex((r, g, b)))

    # Generate 3D plot with Plotly
    fig = go.Figure(data=[go.Scatter3d(
        x=df['x'],
        y=df['y'],
        z=df['z'],
        mode='lines+markers',  # Lines for trajectory, markers for points
        line=dict(
            color=colors_gamma,  # Use gamma-adjusted colors for the line
            width=5,
            colorscale=None  # Custom colors
        ),
        marker=dict(
            size=3,
            color=colors_gamma,  # Colors based on direction with gamma
            symbol='circle'
        )
    )])

    # Customize layout
    fig.update_layout(
        title=name,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        showlegend=True
    )

    # Export to HTML
    fig.write_html(name + '.html', auto_open=True) # Switch for instant view

    fig.show()

In [8]:
def plot_trajectory_3d(df, name):
    # Create a 3D scatter plot (or line plot) using Plotly
    fig = go.Figure()

    # Add the trajectory as a 3D line plot
    fig.add_trace(go.Scatter3d(
        x=df['x'],
        y=df['y'],
        z=df['z'],
        mode='lines',
        line=dict(width=4, color='blue', dash='dash'),
        name='Trajectory'
    ))

    # Update the layout for better visualization
    fig.update_layout(
        title="Interactive 3D Trajectory Plot",
        scene=dict(
            xaxis_title="X axis",
            yaxis_title="Y axis",
            zaxis_title="Z axis",
            aspectmode='cube'  # Ensure axes are scaled proportionally
        ),
        showlegend=True
    )

    # Export to HTML
    fig.write_html(name + '.html', auto_open=True) # Switch for instant view

    # Show it
    fig.show

In [9]:
df_combined

,Unnamed: 0,timeSec,x,y,z
0,0.0,0.000000,26.039776,8.650972,0.0
1,1.0,0.033333,26.956745,8.327490,0.0
2,2.0,0.066667,27.829542,8.114851,0.0
3,3.0,0.100000,28.522152,8.529919,0.0
4,4.0,0.133333,29.582638,8.737701,0.0
...,...,...,...,...,...
134964,26988.0,899.600000,12.522142,24.786190,0.0
134965,26989.0,899.633333,12.600270,24.897448,0.0
134966,26990.0,899.666667,12.624311,25.035426,0.0
134967,26991.0,899.700000,12.800865,25.121545,0.0


In [10]:
plot_trajectory_3d(df_combined, 'df_2D_combined')

In [11]:
plot_3d_trajectory_with_direction(df_combined, 'df_2D_combined')

KeyError: 'direction'